In [72]:
# ===========================
# 01. IMPORTS
# ===========================
import json
import os
import shutil
from delta.tables import DeltaTable
from pyspark.sql import types


# ===========================
# 02. PARAMETERS
# ===========================

# Default parameter values
DEFAULT_CATALOG  = "lab_catalog_bronze"
DEFAULT_SCHEMA   = "digital_insurance"
DEFAULT_TABLE    = "tb_apolice"
DEFAULT_PK_FIELD = "id"

# Read input parameters with default fallback values
dest_catalog = oidlUtils.parameters.getParameter("catalog", DEFAULT_CATALOG)
schema       = oidlUtils.parameters.getParameter("schema", DEFAULT_SCHEMA)
table        = oidlUtils.parameters.getParameter("table", DEFAULT_TABLE)
pk_field     = oidlUtils.parameters.getParameter("pk_field", DEFAULT_PK_FIELD)

# Build derived paths and table names
dest_full_table_name = f"{dest_catalog}.{schema}.{table}"
source_folder        = f"/Volumes/lab_catalog_raw/digital_insurance/FNOL/USER_ADMIN/{table.upper()}"
volume_path          = f"{source_folder}/*.parquet"
bronze_checkpoint_path = f"/Volumes/{dest_catalog}/{schema}/checkpoints/{table}_checkpoint_bronze/"

In [108]:
# ===========================
# 03. HELPERS
# ===========================
def import_schema(tablename):
    """Load the table schema from its JSON definition."""
    with open(
        f"/Workspace/ai-data-platform/src/table-schema/{tablename}.json",
        "r",
    ) as open_file:
        return types.StructType.fromJson(json.load(open_file))


def upsert(df, delta_table_bronze, batch_id):
    CONTROL_COLS = ["rn"]
    count = df.count()

    if count == 0:
      print(f"[batch {batch_id}] No records to process.")
      return


    # Register the micro-batch as a temporary view for SQL processing.
    df.createOrReplaceGlobalTempView(f"view_{table}")

    # Keep only the latest record for each primary key based on op_ts.
    last_value = f"""
        SELECT *
        FROM (
            SELECT *,
                   ROW_NUMBER() OVER (
                       PARTITION BY {pk_field}
                       ORDER BY op_ts DESC
                   ) AS rn
            FROM global_temp.view_{table}
        ) t
        WHERE rn = 1
    """

    # Remove helper columns before merging into the target table.
    df_cdc = spark.sql(last_value).drop(*CONTROL_COLS)

    # Apply CDC operations to the target Delta table.
    (
        delta_table_bronze.alias("b")
          .merge(df_cdc.alias("d"), f"b.{pk_field} = d.{pk_field}")
          .whenMatchedDelete(condition="d.op_type = 'D'")
          .whenMatchedUpdateAll(condition="d.op_type = 'U'")
          .whenNotMatchedInsertAll(condition="d.op_type IN ('I', 'U')")
          .execute()
    )
    print(f"[batch {batch_id}] {dest_full_table_name} table updated - {count} records processed.")

# Load the table schema from the JSON definition.
table_schema = import_schema(table)

In [79]:
# ===========================
# 04. FULL LOAD INGESTION
# ===========================

# Create the target Delta table only if it does not already exist.
if not spark.catalog.tableExists(dest_full_table_name):
    print("Target table does not exist. Starting full load...")

    # Clear the checkpoint directory before the initial load.
    if os.path.isdir(bronze_checkpoint_path):
        shutil.rmtree(bronze_checkpoint_path)

    # Read all Parquet files into a temporary view.
    (
        spark.read
             .parquet(volume_path)
             .createOrReplaceTempView(f"view_{table}")
    )

    # Keep only the latest record for each primary key, excluding deletes.
    query_full = f"""
        SELECT *
        FROM (
            SELECT *,
                   ROW_NUMBER() OVER (
                       PARTITION BY {pk_field}
                       ORDER BY op_ts DESC
                   ) AS rn
            FROM view_{table}
        ) t
        WHERE rn = 1
          AND op_type <> 'D'
    """

    # Remove helper columns before writing to the Delta table.
    full_control_cols = ["current_ts", "pos", "rn"]
    df_query_full = spark.sql(query_full).drop(*full_control_cols)

    # Write the initial dataset to the target Delta table.
    (
        df_query_full.write
                     .format("delta")
                     .mode("overwrite")
                     .saveAsTable(dest_full_table_name)
    )
    print("Full load completed successfully.")
else:
    print("Target table already exists. Skipping full load.")

Target table already exists. Skipping full load.


In [109]:
# ===========================
# 05. INCREMENTAL CDC PROCESSING
# ===========================

# Reference the target Delta table.
delta_table_bronze = DeltaTable.forName(spark, dest_full_table_name)

# Create a streaming DataFrame from the landing Parquet files.
df = (
    spark.readStream
        .format("parquet")
        .schema(table_schema)
        .load(volume_path)
)

# Process each micro-batch and apply CDC changes to the target table.
stream = (
    df.writeStream
      .option("checkpointLocation", bronze_checkpoint_path)
      .foreachBatch(lambda batch_df, batch_id: upsert(batch_df, delta_table_bronze))
      .trigger(availableNow=True) # For scheduled one-time execution. Use trigger(processingTime="1 minute") for continuous streaming.
      .start()
)

# Wait for the streaming job to complete.
stream.awaitTermination()